# Sesión 8 · Series de tiempo y LSTM (versión estudiante)

**Esta es la versión para completar en clase.** Faltan algunos bloques de código marcados con `# TODO`, que corresponden a las partes centrales de la sección (construir las ventanas deslizantes, definir y entrenar la LSTM, y calcular el error comparado con las líneas base).

La solución completa está disponible en `notebooks-soluciones/sesion-07-lstm.ipynb` por si necesita consultarla, pero intente resolver primero por su cuenta.


# Sesión 8 · Series de tiempo y LSTM
**Curso:** Uso de IA para el análisis descriptivo y predictivo en energías renovables

> Ejecute cada celda con `Shift + Enter` y lea la explicación antes de cada paso. Todos los datos son **sintéticos** (simulados) para practicar.

**Objetivo:** convertir una serie horaria en ventanas, calcular líneas base y entrenar una LSTM para predecir la siguiente hora.

## Conceptos: ¿Por qué LSTM para series de tiempo?

### Problema de las redes neuronales clásicas (Dense)
- Una red fully-connected no tiene **memoria**: cada predicción es independiente
- Para predecir la potencia de la hora i, necesitas saber qué pasó en las horas i-1, i-2, ... i-24
- Una red Dense simplemente aplana todo en un vector y pierde el orden temporal

### Solución: Redes Recurrentes (RNN)
- **Recurrencia**: la salida de un paso temporal se alimenta al siguiente
- Teóricamente, pueden "recordar" toda la historia... pero en la práctica olvidan rápido

### LSTM: Long Short-Term Memory
- **Puertas de control** que deciden qué información mantener y qué descartar
  - **Forget gate**: ¿olvido este valor?
  - **Input gate**: ¿agrego este valor nuevo?
  - **Output gate**: ¿saco esto en la salida?
- Resuelve el problema del **vanishing gradient**: puede recordar eventos 100+ pasos atrás
- **Ideal para series de tiempo**: radiación solar, generación de energía, temperaturas tienen patrones repetitivos

**Analogía:** 
- Dense = alumno que ve un examen entero y debe responder en 2 segundos (olvida todo)
- RNN = alumno que va leyendo línea por línea pero la memoria se borra
- LSTM = alumno que toma notas (cell state) y decide cuáles son importantes

## Arquitectura LSTM: Input → Memoria → Output

```
Entrada (batch, tiempo, variables)
     ↓
[LSTM layer 1: 32 unidades]  ← procesa secuencias, genera hidden state
     ↓
[LSTM layer 2: opcional, aprende patrones de más alto nivel]
     ↓
[Dense (fully connected): 1 unidad]  ← convierte el estado a una predicción
     ↓
Salida (batch, 1) ← la potencia de la siguiente hora
```

**Dimensiones en nuestro caso:**
- **Input**: (N muestras, 24 pasos de tiempo, 3 variables=potencia+temp+irr)
- **LSTM output**: cada unidad genera 32 números que representan "características" aprendidas
- **Dense output**: 32 → 1 número (kW de la próxima hora)

**¿Qué aprende?**
- Hora 1: "radiación baja, temperatura baja → potencia mañana será baja"
- Hora 2: "patrón de nubes (irr baja) → va a nublar más"
- Hora 24: "es las 11 AM → próxima hora será pico de generación"

## 1. Serie horaria de potencia solar

## Hiperparámetros: Configuración del modelo

### 1. Número de unidades LSTM
Una unidad LSTM = una "característica" que el modelo aprende a monitorear

```
LSTM(8)   → muy pequeño, puede no capturar la complejidad
LSTM(32)  → balance clásico (nosotros usaremos este)
LSTM(64)  → más grande, más poder de aprendizaje pero más lento
LSTM(128) → muy grande, suele overfitear en datasets pequeños
```

**Regla práctica:** Empieza con 32-64, aumenta si underfits, reduce si overfits.

### 2. Número de capas LSTM
```
1 capa LSTM + Dense      → patrón simple, rápido
2 capas LSTM + Dense     → patrón más complejo, más lento
3+ capas LSTM + Dense    → solo con datasets muy grandes (>10k muestras)
```

Cada capa adicional puede aprender abstracciones más altas:
- Capa 1: aprende ciclo diario (hora del día)
- Capa 2: aprende tendencias (nubes pasando)
- Capa 3: aprende patrones estacionales (invierno vs verano)

### 3. Función de pérdida (Loss)
```
MSE (Mean Squared Error)        → penaliza errores grandes más agresivamente
MAE (Mean Absolute Error)       → penaliza todos los errores por igual
Huber                           → híbrido: MSE para errores pequeños, MAE para grandes
```

**Para energía solar:** MSE es típico, pero MAE es más fácil de interpretar (en kW).

### 4. Optimizador
```
adam           → ajusta velocidad de aprendizaje automáticamente (recomendado)
sgd            → actualización simple pero lenta
rmsprop        → bueno para rnn, menos popular que adam
```

**Adam es el default:** adaptativo, converge rápido, funciona bien en la mayoría de casos.

### 5. Learning rate (tasa de aprendizaje)
Por defecto adam usa 0.001. Valores típicos:

```
0.1       → demasiado alto, el modelo "salta" soluciones
0.01      → alto, puede no converger bien
0.001     → recomendado (default de adam)
0.0001    → bajo, converge lento pero a veces mejor
```

### 6. Batch size
Cuántos ejemplos se procesan antes de actualizar los pesos

```
batch_size=32  → actualiza pesos 30 veces por época, noisy pero rápido
batch_size=64  → balance clásico (nosotros usaremos este)
batch_size=128 → actualiza pesos 8 veces por época, suave pero lento
```

**Regla:** batch_size = potencia de 2, entre 32 y 128.

### 7. EarlyStopping: Parar cuando validación deja de mejorar
```python
EarlyStopping(
    monitor='val_loss',      # monitorea pérdida en validación
    patience=3,              # espera 3 épocas sin mejorar antes de parar
    restore_best_weights=True # recupera los pesos del mejor modelo
)
```

**Importancia:** Sin esto, el modelo puede memorizar el entrenamiento (overfitting).

### 8. Regularización: Dropout
Apaga aleatoriamente neuronas durante el entrenamiento → previene overfitting

```python
tf.keras.layers.LSTM(32, dropout=0.2)  # apaga 20% de las unidades
```

Valores típicos: 0.1 (10%) a 0.3 (30%). No uses >0.5 en LSTM.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# Cargar datos REALES desde GitHub
url = 'https://raw.githubusercontent.com/anclopezg/curso-analisis-datos-energias-renovables/main/docs/datasets/datos_estacion.csv'
df = pd.read_csv(url)
df['fecha_hora'] = pd.to_datetime(df['fecha_hora'])
df = df.set_index('fecha_hora')

# Crear columna de potencia simulada basada en irradiancia y temperatura
# (aproximación de un sistema solar fotovoltaico)
irr = df['irradiancia_wm2'].values
temp = df['temperatura_c'].values
pot = np.clip(0.18 * irr * (1 - 0.004 * (temp - 25)), 0, None)

df['pot'] = pot
df = df[['pot', 'temperatura_c', 'irradiancia_wm2']]
df.columns = ['pot', 'temp', 'irr']

print(f'Dataset: {df.shape} horas desde {df.index[0]} a {df.index[-1]}')
df.head()
plt.figure(figsize=(10,3))
plt.plot(df['pot'].values[:24*5])
plt.title('Potencia solar (5 días)')
plt.ylabel('kW')
plt.xlabel('Horas')
plt.show()

## 2. Líneas base
- **Persistencia:** la próxima hora será igual a la actual.
- **Ayer a la misma hora:** el valor de hace 24 horas.
Dividimos por fecha: 80 % train, 20 % test.

## Overfitting vs Underfitting: Interpretando las gráficas

Cuando ejecutas `modelo.fit(...)`, el modelo genera una gráfica de pérdida (loss) en train y validación.

### Caso 1: Buen ajuste (Good Fit)
```
Loss
  │     train ↘
  │         val ↘
  │  
  └─────────────► Épocas
```
- **Train y validación bajan juntas y siguen cercanas**
- El modelo está aprendiendo patrones reales
- ✅ **Resultado:** buena generalización

### Caso 2: Overfitting (Memorización)
```
Loss
  │     train ↘↘↘
  │         val ╱ (sube)
  │          
  └─────────────► Épocas
```
- **Validación sube mientras train sigue bajando**
- El modelo está memorizando los datos de entrenamiento
- ❌ **Resultado:** predice bien en train, mal en test

**Solución:**
- Aumenta dropout (ej: 0.2 → 0.3)
- Reduce número de unidades (ej: 64 → 32)
- Usa EarlyStopping (como ya hacemos)

### Caso 3: Underfitting (Modelo demasiado simple)
```
Loss
  │     train ↘ (sigue bajando)
  │         val ↘ (sigue bajando)
  │  
  └─────────────► Épocas
```
- **Ambas no mejoran mucho después de 5 épocas**
- El modelo es demasiado simple para los datos
- ❌ **Resultado:** predicciones mediocres en todo

**Solución:**
- Aumenta número de unidades (ej: 32 → 64)
- Agrega más capas LSTM
- Entrena más épocas
- Reduce dropout (ej: 0.3 → 0.1)

## Validación temporal: Por qué NO mezclar train/test en series

### El problema clásico (INCORRECTO)
```python
# ❌ MALO: mezcla pasado y futuro
indices = np.random.shuffle(range(len(df)))
X_train = X[indices[:800]]
X_test = X[indices[800:]]
```

**¿Por qué es malo?**
- Tu test puede contener información del futuro que se coló al train
- El modelo "memoriza" lo que vio en test durante training
- Métricas infladas (parece mejor de lo que realmente es)
- **En producción falla** porque no tiene datos futuros

### La solución correcta: Validación temporal
```python
# ✅ BUENO: respeta el orden temporal
corte = int(len(df) * 0.8)
X_train = X[:corte]      # 0% a 80%: pasado
X_test = X[corte:]       # 80% a 100%: futuro
```

**¿Por qué funciona?**
- Entrena solo con el pasado
- Prueba solo con el futuro (que nunca vio)
- Simula un escenario real: predecir mañana basado en hoy

### Validación temporal k-fold (avanzado)
Para datasets más grandes, puedes usar múltiples "cuts":

```
Fold 1: Train [0-40%]  → Test [40-60%]
Fold 2: Train [0-60%]  → Test [60-80%]
Fold 3: Train [0-80%]  → Test [80-100%]

Promedia el MAE de los 3 folds para una estimación más robusta
```

**En nuestro caso:** Un solo split 80/20 es suficiente con 1000 datos.

## Ventanas deslizantes: Convertir serie temporal en ejemplos

**Problema:** Tenemos 1000 mediciones horarias continuas. ¿Cómo convertirlas en ejemplos para entrenar?

**Solución:** Crear "ventanas" deslizantes.

### Ejemplo con ventana=3 (en la práctica usamos 24):

```
Serie: [10, 20, 30, 40, 50, 60, 70, 80, 90]

Ventana=3:
  Muestra 1: X=[10, 20, 30] → Y=40 (predice la siguiente)
  Muestra 2: X=[20, 30, 40] → Y=50
  Muestra 3: X=[30, 40, 50] → Y=60
  ...
  Muestra 7: X=[70, 80, 90] → Y=? (no hay siguiente)
```

**Ventajas:**
- De 1000 datos → 976 ejemplos de entrenamiento (con V=24)
- Respetar orden temporal: no mezclas pasado con futuro
- El modelo aprende: "dado un patrón de 24 horas, predice la siguiente"

**Parámetro: Tamaño de ventana (V)**
- **V=12**: el modelo ve solo 12 horas atrás
  - ✅ Rápido de entrenar, usa menos memoria
  - ❌ Pierde patrones diarios (ciclo 24h)
- **V=24**: una día completo
  - ✅ Captura el ciclo diario de radiación solar
  - ✅ Balance velocidad-información
- **V=48**: dos días
  - ✅ Ve patrones de tendencia (sube/baja la nubosidad)
  - ❌ Más lento, menos muestras de entrenamiento
- **V=168**: una semana completa
  - ✅ Patrones semanales (ej: consumo más bajo los domingos)
  - ❌ Mucho más pesado, overfitting en datasets pequeños

**Para energía solar:** V=24 suele ser óptimo porque captura el ciclo diario sin ser excesivo.

In [ ]:
from sklearn.metrics import mean_absolute_error
corte = int(len(df)*0.8)
y = df['pot'].values
# la predicción para la hora i usa solo información anterior
persist = y[corte-1:-1]
ayer = y[corte-24:-24]
real = y[corte:]
print('MAE persistencia   :', round(mean_absolute_error(real, persist),2))
print('MAE ayer misma hora:', round(mean_absolute_error(real, ayer),2))

## 3. Ventanas deslizantes
Cada ejemplo: las 24 horas anteriores de 3 variables (potencia, temperatura, irradiancia). La respuesta: la potencia de la hora siguiente. Forma: `(muestras, 24 pasos, 3 variables)`.

In [ ]:
from sklearn.preprocessing import StandardScaler
cols = ['pot','temp','irr']
esc = StandardScaler().fit(df[cols].iloc[:corte])
Z = esc.transform(df[cols])
V = 24  # tamaño de la ventana (horas pasadas que ve el modelo)

# Construir ventanas deslizantes
X, Y = [], []
for i in range(V, len(Z)):
    X.append(Z[i-V:i, :])  # Las V horas anteriores (3 variables)
    Y.append(Z[i, 0])       # Potencia en la hora i (variable 0)

X, Y = np.array(X), np.array(Y)

# Separar train/test respetando el orden temporal
lim = corte - V  # desplazamiento por la ventana
Xtr, Ytr = X[:lim], Y[:lim]
Xte, Yte = X[lim:], Y[lim:]

print('Xtr', Xtr.shape, '| Xte', Xte.shape)

## 4. Modelo clásico de referencia
Una regresión lineal sobre las 72 cifras aplanadas de la ventana. Es el "competidor sencillo" de la LSTM.

In [ ]:
from sklearn.linear_model import Ridge
lin = Ridge(alpha=1.0).fit(Xtr.reshape(len(Xtr),-1), Ytr)
p_lin = lin.predict(Xte.reshape(len(Xte),-1))
def kw(v): return v*esc.scale_[0] + esc.mean_[0]      # volver a kW
print('MAE ridge (kW):', round(mean_absolute_error(kw(Yte), kw(p_lin)),2))

## 5. LSTM con Keras
Una capa LSTM de 32 unidades y una salida. `EarlyStopping` detiene el entrenamiento cuando la validación deja de mejorar. En Colab, TensorFlow ya está instalado.

> Si ve un error de forma, revise que `Xtr` tenga 3 dimensiones.

## Cheat Sheet: Tabla de referencia de hiperparámetros

| Hiperparámetro | Valor por defecto | Rango recomendado | Cuándo cambiar |
|---|---|---|---|
| **Unidades LSTM** | 32 | 8-128 | Si overfits→reduce; si underfits→aumenta |
| **Capas LSTM** | 1 | 1-3 | Solo con >5k muestras y patrón complejo |
| **Dropout** | 0 | 0.1-0.3 | Si overfits→aumenta; si underfits→reduce |
| **Ventana (V)** | 24 | 12-48 | 24 para energía solar (ciclo diario) |
| **Batch size** | 32 | 32-128 | Mantén potencia de 2 |
| **Learning rate** | 0.001 | 0.0001-0.01 | Raro cambiar con Adam |
| **Epochs** | 20 | 5-100+ | EarlyStopping se encarga de parar |
| **EarlyStopping patience** | 3 | 2-5 | 3 es el balance clásico |
| **Validación split** | 0.2 | 0.1-0.3 | 20% de train para validación |

### Receta rápida: "Empieza aquí"
```python
# Configuración conservadora: probablemente funcione
modelo = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(24, 3)),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dense(1)
])
modelo.compile(optimizer='adam', loss='mse')
parada = tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
modelo.fit(Xtr, Ytr, epochs=20, batch_size=64, 
           validation_split=0.2, callbacks=[parada], verbose=1)
```

### Checklist: ¿Cómo sé si mi modelo es bueno?
- ✅ MAE en test < MAE de línea base (persistencia, ayer)
- ✅ Gráfica de loss: train y val bajan juntas
- ✅ No tardó >5 minutos en entrenar
- ✅ La predicción no es un "promedio plano"
- ✅ Captura picos (mañanas con mucha radiación)

### Tips para debug rápido
1. **Modelo muy lento:** reduce unidades (32→16) o batch_size (64→128)
2. **MAE no mejora:** overfitting seguro, aumenta dropout
3. **MAE igual que persistencia:** modelo muy simple, aumenta unidades
4. **NaN en loss:** learning rate demasiado alto (raro con Adam)
5. **GPU out of memory:** reduce batch_size o ventana V

In [ ]:
p_lstm = None
try:
    import tensorflow as tf
    tf.random.set_seed(0)
    
    modelo = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(V, len(cols))),
        tf.keras.layers.LSTM(32),
        tf.keras.layers.Dense(1)
    ])
    modelo.compile(optimizer='adam', loss='mse')
    
    parada = tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
    
    hist = modelo.fit(Xtr, Ytr, epochs=20, batch_size=64,
                      validation_split=0.2, callbacks=[parada], verbose=0)
    
    p_lstm = modelo.predict(Xte, verbose=0).ravel()
    
    print('MAE LSTM (kW):', round(mean_absolute_error(kw(Yte), kw(p_lstm)),2))
    plt.plot(hist.history['loss'], label='train')
    plt.plot(hist.history['val_loss'], label='val')
    plt.legend()
    plt.show()
except ImportError:
    print('TensorFlow no está instalado aquí. Use Google Colab para este paso.')

In [ ]:
resultados_modelos = {}

try:
    import tensorflow as tf
    import time
    tf.random.set_seed(0)
    
    configs = [
        {'nombre': '8 unidades', 'unidades': 8, 'capas': 1, 'dropout': 0.0},
        {'nombre': '32 unidades', 'unidades': 32, 'capas': 1, 'dropout': 0.0},
        {'nombre': '64 unidades + dropout', 'unidades': 64, 'capas': 1, 'dropout': 0.2},
        {'nombre': 'Dos capas (32+32)', 'unidades': [32, 32], 'capas': 2, 'dropout': 0.0},
    ]
    
    for config in configs:
        print(f"\n{'='*50}")
        print(f"Entrenando: {config['nombre']}")
        print(f"{'='*50}")
        
        inicio = time.time()
        
        if config['capas'] == 1:
            modelo = tf.keras.Sequential([
                tf.keras.layers.Input(shape=(V, len(cols))),
                tf.keras.layers.LSTM(config['unidades'], dropout=config['dropout']),
                tf.keras.layers.Dense(1)
            ])
        else:  # 2 capas
            modelo = tf.keras.Sequential([
                tf.keras.layers.Input(shape=(V, len(cols))),
                tf.keras.layers.LSTM(config['unidades'][0], return_sequences=True),
                tf.keras.layers.LSTM(config['unidades'][1]),
                tf.keras.layers.Dense(1)
            ])
        
        modelo.compile(optimizer='adam', loss='mse')
        parada = tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
        
        hist = modelo.fit(Xtr, Ytr, epochs=20, batch_size=64,
                          validation_split=0.2, callbacks=[parada], verbose=0)
        
        p_pred = modelo.predict(Xte, verbose=0).ravel()
        mae = mean_absolute_error(kw(Yte), kw(p_pred))
        tiempo = time.time() - inicio
        
        resultados_modelos[config['nombre']] = {
            'MAE': mae,
            'Tiempo': tiempo,
            'Épocas': len(hist.history['loss'])
        }
        
        print(f"MAE: {mae:.2f} kW")
        print(f"Tiempo: {tiempo:.1f}s")
        print(f"Épocas completadas: {len(hist.history['loss'])}")
    
    # Tabla comparativa
    print(f"\n{'='*50}")
    print("COMPARACIÓN DE MODELOS")
    print(f"{'='*50}")
    df_resultados = pd.DataFrame(resultados_modelos).T
    df_resultados = df_resultados.round(2)
    print(df_resultados)
    
    # Gráfico
    df_resultados['MAE'].sort_values().plot(kind='barh', color='steelblue')
    plt.xlabel('MAE (kW)')
    plt.title('Comparación de modelos: MAE en datos de test')
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print('TensorFlow no está disponible. Ejecuta en Google Colab.')

## Experimento: Comparar modelos con diferentes hiperparámetros

**Pregunta:** ¿Qué configuración es mejor?

Vamos a entrenar 4 modelos y comparar su MAE:
1. **Pequeño** (8 unidades, sin dropout)
2. **Mediano** (32 unidades, sin dropout) ← el que ya entrenamos
3. **Grande** (64 unidades, con dropout=0.2)
4. **Profundo** (dos capas LSTM de 32 unidades cada una)

Esto te ayuda a entender el trade-off: **precisión vs velocidad vs overfitting**

## 6. Tabla comparativa
Llene la tabla con los MAE obtenidos. La LSTM solo se justifica si supera a las líneas base y al modelo lineal.

In [ ]:
# Armar tabla comparativa
res = {
    'Persistencia': mean_absolute_error(real, persist),
    'Ayer misma hora': mean_absolute_error(real, ayer),
    'Ridge (ventana)': mean_absolute_error(kw(Yte), kw(p_lin))
}

if p_lstm is not None:
    res['LSTM'] = mean_absolute_error(kw(Yte), kw(p_lstm))

tabla = pd.Series(res).round(2).sort_values()
print(tabla)
tabla.plot.barh()
plt.xlabel('MAE (kW)')
plt.show()

## 7. Real contra predicho

In [ ]:
n = 24*4
plt.figure(figsize=(10,3.5))
plt.plot(kw(Yte)[:n], label='real')
plt.plot(kw(p_lin)[:n], label='ridge')
if p_lstm is not None: plt.plot(kw(p_lstm)[:n], label='LSTM')
plt.legend(); plt.show()

## 8. Ejercicios
1. Cambie la ventana `V` a 12 y a 48. ¿Cómo cambia el MAE?
2. Quite la irradiancia de `cols`. ¿Qué pasa?
3. Cambie las unidades de la LSTM a 8 y a 128.
4. ¿Qué modelo elegiría y por qué? Considere precisión, simplicidad y tiempo.